# 05 - Reward function and Model Ranking

Delphos evaluates models using a reinforcement learning concept known as a *reward function*. While the agent uses its own internal reward (based on improvements in Log-Likelihood) to guide its search, as a choice modeller, you will want to evaluate and rank the final candidates using standard statistical metrics.

In this notebook, we will:
- Understand the internal reward function used by Delphos.
- Rank candidate models using AIC, BIC, and Adjusted Rho-squared.
- Compare competing specifications.


In [1]:
import delphos as dp
import pandas as pd

dataset = dp.load_dataset("dataset_4")
agent = dp.load_agent()

# Propose and estimate 3 models
models = agent.propose(dataset, n_models=3, estimate=True)
results_df = models.to_dataframe()


### 1. How Delphos calculates reward

Internally, the `delphos.env.reward.reward_function` relies heavily on the improvement in Log-Likelihood (`LLout`) relative to a null or linear-additive baseline model (`LL0`). 

Because of this, Delphos has learned a policy that strongly prefers specifications that maximize `LLout` while avoiding bloated models that fail to converge (which return a reward of `-1.0`).


### 2. Ranking models by BIC or AIC

Once you have generated the proposals, you can define your own evaluation criteria. For instance, choice modellers often prefer the **Bayesian Information Criterion (BIC)** to penalize overly complex models.

Let's rank our results by BIC:

In [2]:
# Sort models by BIC (lower is better)
best_by_bic = results_df.sort_values("BIC", ascending=True)

print("Top model by BIC:")
best_by_bic[["specification_key", "BIC", "AIC", "LLout", "nFreeParams"]].head(1)


Top model by BIC:


,specification_key,BIC,AIC,LLout,nFreeParams
0,1110_2124_3211_4111_5000_6126_7000,8251.932239,8159.590766,-4065.795383,14.0


If you prefer to evaluate based on predictive power, you might rank by **Adjusted Rho-squared** instead:

In [3]:
# Sort models by Adjusted Rho-squared (higher is better)
best_by_rho2 = results_df.sort_values("adjRho2_C", ascending=False)

print("Top model by Adjusted Rho-squared:")
best_by_rho2[["specification_key", "adjRho2_C", "BIC", "nFreeParams"]].head(1)


Top model by Adjusted Rho-squared:


,specification_key,adjRho2_C,BIC,nFreeParams
0,1110_2124_3211_4111_5000_6126_7000,0.132929,8251.932239,14.0


By separating Delphos's internal search reward from your external model ranking, you get the best of both worlds: a powerful heuristic search combined with rigorous statistical model selection.